# NFL Data — Colab Runner

Thin execution shell. All logic lives in the `nfl_data` package in this repo — edit that in Claude Code, not here.

This notebook: clones the repo fresh → installs it → authenticates to GCP → runs the pipeline → writes to BigQuery.

By default, writes to two BigQuery datasets in `ff-python-api`:
- `nflreadpy` — raw nflreadpy pulls (players, player_stats, snap_counts, nextgen_stats, ff_opportunity), replaced each run
- `dynasty` — `yprr_proxy`, a derived route-share proxy table

The Sleeper-based `dynasty_tycoon.player_auction_values` valuation stage is **not** run by default — see step 6 to run it explicitly.

## 1. Clone the repo

Set `REPO_URL` once. The repo is public, so no token is needed.

In [ ]:
REPO_URL = "https://github.com/nashstallings/fantasy_football.git"  # update this
REPO_DIR = "fantasy_football"

import subprocess

subprocess.run(["rm", "-rf", REPO_DIR], check=True)
subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

## 2. Install the package

In [ ]:
import subprocess

subprocess.run(["pip", "install", "-q", "-e", "."], cwd=REPO_DIR, check=True)

## 3. Auth

GCP auth via Colab's built-in account.

In [ ]:
from google.colab import auth
auth.authenticate_user()

## 4. Run the pipeline

In [ ]:
import os
import sys

src_dir = os.path.abspath(f"{REPO_DIR}/src")
assert os.path.isdir(os.path.join(src_dir, "nfl_data")), (
    f"{src_dir}/nfl_data not found -- re-run steps 1-2 above (clone + install) first"
)
sys.path.insert(0, src_dir)

import nfl_data

tables = nfl_data.run(write_to_bq=True)
for name, df in tables.items():
    print(name, df.shape)

## 5. Verify

Row counts per dataset (only the two the default pipeline writes to — see step 6 for `dynasty_tycoon`).

In [ ]:
from nfl_data.bigquery_io import verify
from nfl_data import config

for dataset_id in (config.NFLREADPY_DATASET_ID, config.YPRR_DATASET_ID):
    print(f"--- {dataset_id} ---")
    print(verify(config.PROJECT_ID, dataset_id))

## 6. Run a single stage (optional)

Each stage can be run independently — useful if you only need to refresh one dataset, or want to iterate on `season`/`seasons` without re-running everything.

In [ ]:
# from nfl_data import run_nflreadpy_tables, run_yprr, run_auction_values

# run_nflreadpy_tables(season=2025)
# run_yprr(seasons=list(range(2012, 2026)))
# run_auction_values(season=2026)